# 🔬 Combined Experiment 1 & 2: Single-Run LOPO-CV Benchmark (72 Factorial Pipelines)
**Research Group**: Group A01 — Advanced Software Engineering (Master of Computer Science, ITS)  
**Benchmark Objective**: Evaluates the **Tri-Paradigm Data Cleaning Framework (C1: Row Discarding vs. C2: Project Discarding vs. C3: Group-Aware Median Imputation)** systematically crossed with **3 Scalers (S1/S2/S3)**, **4 Resamplers (B1/B2/B3/B4)**, and **2 Classifiers (M1: RF CPU / M2: XGBoost CUDA GPU)** ($3 \times 3 \times 4 \times 2 = \mathbf{72\text{ Pipelines}}$).  
**Validation Protocol**: Leave-One-Project-Out Cross-Validation (LOPO-CV, Zero Data Leakage) across 24 Java Repositories (Total: **1,608 Evaluated Model Folds**).

## 1. Environment & Hardware Diagnostics (RTX 4050 CUDA Acceleration)

In [ ]:
import sys
import os
from pathlib import Path

src_dir = str(Path("../src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import check_gpu
check_gpu.check_nvidia_smi()
check_gpu.check_python_environment()
check_gpu.check_xgboost_gpu()

## 2. Ingestion & Comparison of Tri-Paradigm Datasets (C1, C2, C3)

In [ ]:
from data_loader import load_dataset

df_c1, feat_c1, proj_c1 = load_dataset('C1')
df_c2, feat_c2, proj_c2 = load_dataset('C2')
df_c3, feat_c3, proj_c3 = load_dataset('C3')

print("\n" + "="*70)
print("TRI-PARADIGM DATASET INGESTION SUMMARY")
print("="*70)
print(f"  C1 (Row-Level Discarding)     : {len(df_c1):>6,} rows | {len(proj_c1):>2} projects | {df_c1['flaky'].sum():>3} flaky ({df_c1['flaky'].mean()*100:.2f}%)")
print(f"  C2 (Project-Level Discarding) : {len(df_c2):>6,} rows | {len(proj_c2):>2} projects | {df_c2['flaky'].sum():>3} flaky ({df_c2['flaky'].mean()*100:.2f}%)")
print(f"  C3 (Group-Aware Imputation)   : {len(df_c3):>6,} rows | {len(proj_c3):>2} projects | {df_c3['flaky'].sum():>3} flaky ({df_c3['flaky'].mean()*100:.2f}%)")
print("="*70)

## 3. Single-Run LOPO-CV Execution (72 Machine Learning Pipelines = 1,608 Model Folds)
- If cached benchmark summary (`results/summary_metrics_72_pipelines.csv`) exists, loads immediately.
- Set `force_rerun = True` to execute all 1,608 model folds from scratch (~20–25 minutes with CUDA GPU acceleration).

In [ ]:
import pandas as pd
from lopo_runner import run_all_72_pipelines

summary_file = Path("../results/summary_metrics_72_pipelines.csv")
force_rerun = False  # Set to True to re-execute benchmark from scratch

if not summary_file.exists() or force_rerun:
    print("🚀 Launching Single-Run LOPO-CV Benchmark across all 72 Pipelines...")
    df_summary = run_all_72_pipelines(output_dir="../results", use_gpu=True, quick_test=False)
else:
    print(f"[OK] Discovered existing summary metrics: {summary_file.resolve()}")
    df_summary = pd.read_csv(summary_file)

print(f"Total Evaluated Pipelines: {len(df_summary)}")
display(df_summary.head(10))

## 4. Top 10 High-Performing Pipelines (Ranked by Mean F1)

In [ ]:
top10 = df_summary.sort_values(by="Mean_F1", ascending=False).head(10)
cols = ["Pipeline_ID", "Cleaning", "Scaling", "Imbalance", "Model", "Mean_F1", "Mean_Recall", "Mean_Precision", "Mean_PR_AUC", "Mean_ROC_AUC", "Mean_MCC"]
display(top10[[c for c in cols if c in top10.columns]])

## 5. Statistical Significance Tests (Kruskal-Wallis Paradigm Comparison)

In [ ]:
from statistical_tests import run_cleaning_comparison_test
run_cleaning_comparison_test(str(summary_file))